# Title of the Project

## Food Delivery ETA Prediction using Regression and Machine Learning

# 1. Name and Roll Number

**Name:** Khushi Gupta               **Roll Number:** 24csu100

**Name:** Madhavan Dixit             **Roll Number:** 24csu118 

# 2. Brief Description of Dataset

The dataset contains historical food delivery order information.

Each row represents one food delivery order. The dataset contains information
about the order, restaurant/store, food category, order amount, number of
delivery partners, outstanding orders, estimated preparation duration and
estimated driving duration.

The objective of this project is to predict the actual food delivery time
using machine learning techniques.

The dataset contains 197,428 records and 16 attributes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

Load dataset


In [5]:
df=pd.read_csv("historical_data.csv")
df.shape
df.info()
df.head()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     196441 non-null  float64
 1   created_at                                    197428 non-null  object 
 2   actual_delivery_time                          197421 non-null  object 
 3   store_id                                      197428 non-null  int64  
 4   store_primary_category                        192668 non-null  object 
 5   order_protocol                                196433 non-null  float64
 6   total_items                                   197428 non-null  int64  
 7   subtotal                                      197428 non-null  int64  
 8   num_distinct_items                            197428 non-null  int64  
 9   min_item_price                                19

,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
count,196441.000000,197428.000000,196433.000000,197428.000000,197428.000000,197428.000000,197428.000000,197428.000000,181166.000000,181166.000000,181166.000000,197428.000000,196902.000000
mean,2.978706,3530.510272,2.882352,3.196391,2682.331402,2.670791,686.218470,1159.588630,44.808093,41.739747,58.050065,308.560179,545.358935
std,1.524867,2053.496711,1.503771,2.666546,1823.093688,1.630255,522.038648,558.411377,34.526783,32.145733,52.661830,90.139653,219.352902
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,-86.000000,0.000000,-4.000000,-5.000000,-6.000000,0.000000,0.000000
25%,2.000000,1686.000000,1.000000,2.000000,1400.000000,1.000000,299.000000,800.000000,17.000000,15.000000,17.000000,251.000000,382.000000
50%,3.000000,3592.000000,3.000000,3.000000,2200.000000,2.000000,595.000000,1095.000000,37.000000,34.000000,41.000000,251.000000,544.000000
75%,4.000000,5299.000000,4.000000,4.000000,3395.000000,3.000000,949.000000,1395.000000,65.000000,62.000000,85.000000,446.000000,702.000000
max,6.000000,6987.000000,7.000000,411.000000,27100.000000,20.000000,14700.000000,14700.000000,171.000000,154.000000,285.000000,2715.000000,2088.000000


# DATA PREPROCESSING

4.1 Check missing values

In [7]:
df.isnull().sum()

market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526
dtype: int64

4.2 converting dates and creating target values

In [8]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])
df['delivery_time_minutes'] = (
    df['actual_delivery_time'] - df['created_at']
).dt.total_seconds() / 60

df[['created_at', 'actual_delivery_time', 'delivery_time_minutes']].head()

,created_at,actual_delivery_time,delivery_time_minutes
0,2015-02-06 22:24:17,2015-02-06 23:27:16,62.983333
1,2015-02-10 21:49:25,2015-02-10 22:56:29,67.066667
2,2015-01-22 20:39:28,2015-01-22 21:09:09,29.683333
3,2015-02-03 21:21:45,2015-02-03 22:13:00,51.250000
4,2015-02-15 02:40:36,2015-02-15 03:20:26,39.833333


4.3 Remove rows where target doesn't exist


In [9]:
df = df.dropna(subset=['delivery_time_minutes'])

4.4 Check strange values

In [10]:
df["delivery_time_minutes"].describe()

count    197421.000000
mean         48.470956
std         320.493482
min           1.683333
25%          35.066667
50%          44.333333
75%          56.350000
max      141947.650000
Name: delivery_time_minutes, dtype: float64

In [11]:
df = df[
    (df['delivery_time_minutes'] >= 5) &
    (df['delivery_time_minutes'] <= 120)
]

df["delivery_time_minutes"].describe()

count    196328.000000
mean         47.098376
std          16.930512
min           5.350000
25%          35.016667
50%          44.216667
75%          56.083333
max         119.933333
Name: delivery_time_minutes, dtype: float64

4.5 Create useful time features

In [12]:
df['order_hour'] = df['created_at'].dt.hour
df['order_day'] = df['created_at'].dt.dayofweek
df['order_month'] = df['created_at'].dt.month
df['is_weekend'] = (
    df['order_day'] >= 5
).astype(int)
df[['created_at', 'order_hour', 'order_day',
    'order_month', 'is_weekend']].head()

,created_at,order_hour,order_day,order_month,is_weekend
0,2015-02-06 22:24:17,22,4,2,0
1,2015-02-10 21:49:25,21,1,2,0
2,2015-01-22 20:39:28,20,3,1,0
3,2015-02-03 21:21:45,21,1,2,0
4,2015-02-15 02:40:36,2,6,2,1


4.6 Remove columns we shouldn't use

In [13]:
X = df.drop(
    columns=[
        'actual_delivery_time',
        'created_at',
        'delivery_time_minutes'
    ]
)

y = df['delivery_time_minutes']